### **Práctica 9 - Pipeline para clasificación de textos - Sentence Embedding**
- Lectura y análisis preliminar del dataset.
- Preprocesamiento de textos.
- Procesamiento de la variables de salida.
- Representación vectorial del lenguaje.
    - SentenceEmbedding
- Balance y partición del conjunto de datos.
- Entrenamiento y evaluación de modelos.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import SMOTE
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import classification_report, accuracy_score

import nltk
import string
from nltk.corpus import stopwords

nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

#### **Lectura del Dataset**

In [ ]:
# Lectura del dataset

# Vista de los datos
df.head()

#### **Preprocesamiento ligero de Textos**

In [ ]:
def clean_for_embedding(text):
    text = text.lower()
    text = ''.join([c for c in text if c not in string.punctuation])
    return text

df['clean_text'] = df['text'].apply(clean_for_embedding)

In [ ]:
df['clean_text']

#### **Procesamiento de variable de salida**

In [ ]:
# Convertir sentimiento a números
label_encoder = LabelEncoder()
df['label'] = label_encoder.fit_transform(df['airline_sentiment'])  # 0: negative, 1: neutral, 2: positive

In [ ]:
df['airline_sentiment'].value_counts()

In [ ]:
df['label'].value_counts()

In [ ]:
y = df['label']

#### **Generar Embeddings**

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')
X_embed = model.encode(df['clean_text'], show_progress_bar=True)

In [ ]:
X_embed.shape

In [ ]:
X_embed

#### **Balance del conjunto de datos**

In [ ]:
sns.countplot(data=df, y=y)

In [ ]:
# SMOTE para balancear
smote = SMOTE()

X_res, y_res = smote.fit_resample(X_embed, y)

In [ ]:
sns.countplot(y=y_res)

#### **Separar el dataset en train/test**

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_res, y_res, test_size=0.3)

#### **Definir modelos de AA**

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(n_estimators=100),
    "Naive Bayes": GaussianNB()
}

#### **Entrenar y evaluar modelos**


In [ ]:
# Guardar los resultados
resultados_accuracy = []

def evaluate_models(X_train, X_test, y_train, y_test, vectorizer_name):
    print(f"\n Resultados con {vectorizer_name}:\n" + "-"*40)
    for name, model in models.items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        acc = accuracy_score(y_test, y_pred)
        print(f"\n Modelo: {name}")
        print("Accuracy:", acc)
        print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

         # Guardar en la lista global
        resultados_accuracy.append({
            'Modelo': name,
            'Vectorización': vectorizer_name,
            'Accuracy': acc
        })

In [ ]:
# Evaluar 
evaluate_models(X_train, X_test, y_train, y_test, "Embedding")

In [ ]:
df_results = pd.DataFrame(resultados_accuracy)

plt.figure(figsize=(10, 6))
sns.barplot(data=df_results, x='Modelo', y='Accuracy', hue='Vectorización', palette='Set2')
plt.title('Comparación de Accuracy por Modelo y Vectorización')
plt.ylim(0, 1)
plt.tight_layout()
plt.show()